In [ ]:
# ============================================================
# Cell 1: Leakage-Proof Temporal Feature Extraction & Split
# (Modular: src/nids/temporal_dataset.py)
# ============================================================
import os, sys
from pathlib import Path

PROJECT = "/kaggle/working/CyberShield-BigData"
SOURCE = "/kaggle/input/datasets/mennatullahbadawy/cybershield-bigdata-project"

# على Kaggle: انسخ المشروع المرفوع — محلياً: استخدم مجلد الريبو نفسه
if os.path.exists("/kaggle/working"):
    import shutil
    if os.path.exists(PROJECT):
        shutil.rmtree(PROJECT)
    shutil.copytree(SOURCE, PROJECT)
    os.chdir(PROJECT)
else:
    PROJECT = str(Path.cwd())

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from src.nids.temporal_dataset import build_temporal_dataset

RAW = os.path.join(PROJECT, "data/feature_store/nids_features_latest")
meta = build_temporal_dataset(output_dir=RAW)
print("✅ Cell 1 complete successfully.")


In [ ]:
# ============================================================
# Cell 2: Target Sweet-Spot Benchmark (Precision >= 92% & Recall >= 90%)
# (Modular: src/nids/benchmark.py)
# ============================================================
import os
from src.nids.benchmark import run_benchmark

OUT = "/kaggle/working/benchmark_results" if os.path.exists("/kaggle/working") else "./benchmark_results"

bench = run_benchmark(store_dir=RAW, out_dir=OUT)

results = bench["results"]
df = bench["df"]
val_predictions = bench["val_predictions"]
test_predictions = bench["test_predictions"]
models = bench["models"]
scaler = bench["scaler"]
X_train_t, y_train_t = bench["X_train_t"], bench["y_train_t"]
X_tr_tab, y_train = bench["X_tr_tab"], bench["y_train"]
y_val, y_test = bench["y_val"], bench["y_test"]
device = bench["device"]
print("✅ Benchmark complete!")


In [ ]:
# ============================================================
# Cell 3: Robust Generalization & IEEE/ACM Benchmark Diagnostic Suite
# (Modular: src/nids/diagnostics.py)
# ============================================================
from src.nids.diagnostics import run_diagnostics

df_diag = run_diagnostics(
    models=models,
    test_predictions=test_predictions,
    results=results,
    X_tr_tab=X_tr_tab,
    X_train_t=X_train_t,
    y_train_t=y_train_t,
    y_test=y_test,
    out_dir=OUT,
    device=device,
)


In [ ]:
# ============================================================
# Cell 4: Publication-Quality Visualization & Confusion Matrix Suite
# (Modular: src/nids/figures.py)
# ============================================================
from src.nids.figures import generate_all_figures

fig_paths = generate_all_figures(
    test_predictions=test_predictions,
    results=results,
    y_test=y_test,
    df=df,
    out_dir=OUT,
)


In [ ]:
# ============================================================
# Cell 5: Professional PDF Report + Full API
# (Modular: src/nids/{api, soc_report, threat_kb, xai}.py)
# ============================================================
import os
import numpy as np
import torch
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix, roc_auc_score

from src.models.deep_learning_models import MambaNIDS
from src.nids.threat_kb import THREAT_KB, HybridThreatRetriever
from src.nids.xai import extract_xai
from src.nids.api import create_app, serve_in_background
from src.nids.soc_report import generate_soc_report
from src.nids.temporal_dataset import load_metadata

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_chk = np.load(os.path.join(RAW, "X_train.npy"), mmap_mode='r')
API_NF, API_SL = X_chk.shape[2], X_chk.shape[1]
print(f"⚡ Input: ({API_SL}, {API_NF})")

mamba = MambaNIDS(input_dim=API_NF, d_model=64, d_state=16,
                  num_layers=2, num_classes=2, dropout=0.2).to(device)
mpath = os.path.join(OUT, "mamba_ssm_best.pt")
if os.path.exists(mpath):
    ckpt = torch.load(mpath, map_location=device, weights_only=False)
    try:
        mamba.load_state_dict(ckpt['model_state_dict'])
        print("✅ Model loaded")
    except Exception:
        print("⚠️ Fresh model")
mamba.eval()

scaler_api = RobustScaler(quantile_range=(5.0, 95.0))
scaler_api.fit(np.load(os.path.join(RAW, "X_train.npy")).reshape(-1, API_NF))

# أسماء الميزات الحقيقية من الـ metadata (إصلاح عدم التطابق في النوت بوك الأصلية)
try:
    FEATURE_NAMES = load_metadata(RAW).get("feature_names") or []
except Exception:
    FEATURE_NAMES = []
while len(FEATURE_NAMES) < API_NF:
    FEATURE_NAMES.append(f"Feature_{len(FEATURE_NAMES)}")

retriever = HybridThreatRetriever()

# ── Batch inference: 50 attacks + 50 benign ──
print("\n🧪 Running batch inference...")
X_test_api = np.load(os.path.join(RAW, "X_test.npy"))
y_test_api = np.load(os.path.join(RAW, "y_test.npy"))
attack_idx = np.where(y_test_api == 1)[0]
benign_idx = np.where(y_test_api == 0)[0]
n_test = 50
test_indices = np.concatenate([
    np.random.choice(attack_idx, min(n_test, len(attack_idx)), replace=False),
    np.random.choice(benign_idx, min(n_test, len(benign_idx)), replace=False),
])
test_true = y_test_api[test_indices]
test_probs, test_preds = [], []
batch_size = 32
with torch.no_grad():
    for i in range(0, len(test_indices), batch_size):
        batch_idx = test_indices[i:i + batch_size]
        batch = X_test_api[batch_idx]
        flat = batch.reshape(-1, API_NF)
        scaled = np.clip(scaler_api.transform(flat), -15, 15).reshape(len(batch_idx), API_SL, API_NF)
        x = torch.tensor(scaled, dtype=torch.float32, device=device)
        probs = torch.softmax(mamba(x), 1)[:, 1].cpu().numpy()
        test_probs.extend(probs)
        test_preds.extend((probs >= 0.03).astype(int))
test_probs = np.array(test_probs)
test_preds = np.array(test_preds)
cm = confusion_matrix(test_true, test_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
roc = roc_auc_score(test_true, test_probs)
print(f"   TP={tp} | FP={fp} | TN={tn} | FN={fn} | ROC-AUC={roc:.4f}")

# XAI لأقوى هجمة + Threat types
best_attack = test_indices[np.argmax(test_probs)]
x_atk = X_test_api[best_attack]
flat = x_atk.reshape(-1, API_NF)
scaled = np.clip(scaler_api.transform(flat), -15, 15).reshape(1, API_SL, API_NF)
x_tensor = torch.tensor(scaled, dtype=torch.float32, device=device)
xai_features = extract_xai(mamba, x_tensor, FEATURE_NAMES, n=5)

print("\n📡 Collecting all threat types...")
threat_types_detected = {}
for i, idx in enumerate(test_indices):
    if test_preds[i] == 1:
        x_i = X_test_api[idx]
        flat_i = x_i.reshape(-1, API_NF)
        scaled_i = np.clip(scaler_api.transform(flat_i), -15, 15).reshape(1, API_SL, API_NF)
        x_t = torch.tensor(scaled_i, dtype=torch.float32, device=device)
        xf = extract_xai(mamba, x_t, FEATURE_NAMES, n=3)
        qt = " ".join(f["name"] for f in xf).lower().replace("_", " ")
        rag = retriever.retrieve(qt)
        tid = rag["doc"]["threat_id"]
        if tid not in threat_types_detected:
            threat_types_detected[tid] = {
                "title": rag["doc"]["title"],
                "tactic": rag["doc"]["tactic"],
                "playbook": rag["doc"]["playbook"],
                "indicators": rag["doc"].get("indicators", ""),
                "iptables": rag["doc"].get("iptables", []),
                "count": 0,
                "hybrid_score": rag["hybrid_score"],
            }
        threat_types_detected[tid]["count"] += 1
print(f"   Detected {len(threat_types_detected)} threat types:")
for tid, info in threat_types_detected.items():
    print(f"   • {tid}: {info['title']} ({info['count']} occurrences)")

# ── FastAPI ──
app = create_app(
    mamba=mamba, scaler=scaler_api, feature_names=FEATURE_NAMES,
    api_sl=API_SL, api_nf=API_NF, device=device,
    batch_stats={"tp": tp, "fp": fp, "tn": tn, "fn": fn, "roc": roc},
    threat_types_detected=threat_types_detected,
    retriever=retriever,
)
PORT = serve_in_background(app)

# ── PDF Report ──
PDF_PATH = "/kaggle/working/CyberShield_SOC_Report.pdf" if os.path.exists("/kaggle/working") else "./CyberShield_SOC_Report.pdf"
print("\n📄 Generating Professional PDF Report...")
generate_soc_report(
    pdf_path=PDF_PATH, tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
    roc=float(roc), n_samples=len(test_indices), cm=cm,
    xai_features=xai_features, threat_types_detected=threat_types_detected,
    threat_kb=THREAT_KB, port=PORT, results=results,
)
print(f"   Threat Types: {len(threat_types_detected)}")
print(f"   API: http://localhost:{PORT}")


In [ ]:
# ============================================================
# Cell 6: Bot dependencies (merged old cells 5+6)
# ============================================================
!pip install -q reportlab python-telegram-bot nest_asyncio requests


In [ ]:
# ============================================================
# Cell 7: SOC Telegram Live Monitor
# (Modular: src/nids/telegram_bot.py — includes reconstructed cells)
# ============================================================
import os
import numpy as np
import nest_asyncio
nest_asyncio.apply()

from src.nids.telegram_bot import run_monitor

X_test_raw = np.load(os.path.join(RAW, "X_test.npy"))
y_test_bot = np.load(os.path.join(RAW, "y_test.npy"))

# ضع التوكن في Environment (Kaggle Secrets) وليس في الكود:
# os.environ["CYBERSHIELD_BOT_TOKEN"] = "..."
# os.environ["CYBERSHIELD_ADMIN_CHAT_ID"] = "..."

await run_monitor(
    X_test_raw=X_test_raw,
    y_test=y_test_bot,
    model=mamba,
    scaler=scaler_api,
    feature_names=FEATURE_NAMES,
    retriever=retriever,
    device=device,
    n_demo_attacks=2,
    poll_seconds=120,
)
